# shap vs. shapiq TreeSHAP-IQ — Census Income Benchmark

This notebook benchmarks **shap** (`shap.TreeExplainer`) against
**shapiq**'s `TreeExplainer` (which implements the TreeSHAP-IQ algorithm,
Muschalik et al., AAAI 2024) on the UCI Adult / Census Income dataset.

**shap** (Lundberg & Lee, NeurIPS 2017) is the original Python package
implementing TreeSHAP. Its `TreeExplainer` computes exact Shapley values
and pairwise Shapley interaction values for tree ensembles.

## What we measure

| Quantity | shap | shapiq |
|---|---|---|
| Shapley values (SV) | `shap.TreeExplainer(model).shap_values(X)` | `max_order=1, index="SV"` |
| Pairwise interactions | `shap.TreeExplainer(model).shap_interaction_values(X)` | `max_order=2, index="k-SII"` |

A few notes on the comparison:

* **For order-1 Shapley values**, both libraries compute the standard
  path-dependent TreeSHAP and should agree numerically up to floating-point noise.
  We verify this.
* **For interactions, the definitions differ.** shap returns the TreeSHAP
  interaction values of Lundberg et al. (2020): an $n \times n$ matrix with
  main effects on the diagonal. shapiq returns k-SII values (Bordt & von
  Luxburg, 2023), which aggregate higher-order terms differently. The values
  are *not* expected to match numerically, but the timing comparison is valid.
* **Perturbation mode**: both default to tree-path-dependent perturbation
  (no background dataset).

## Parallelism

* **shapiq** uses joblib: `shapiq.TreeExplainer(...).explain_X(X, n_jobs=-1)`.
* **shap** uses C++ extensions internally with OpenMP.

## References

* shap / TreeSHAP: Lundberg & Lee, NeurIPS 2017; Lundberg et al., Nature Machine Intelligence 2020
* shapiq / TreeSHAP-IQ: Muschalik et al., AAAI 2024

## 1. Setup

In [1]:
# Install (uncomment if needed)
%pip install shap shapiq xgboost lightgbm scikit-learn pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
import shap
import shapiq

from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

print("shap:   ", shap.__version__)
print("shapiq: ", shapiq.__version__)

shap:    0.51.0
shapiq:  None


## 2. Load and pre-process Census Income data

We fetch the Adult dataset from OpenML (version 2). Pre-processing:
one-hot encode categoricals, drop rows with missing-value markers,
cast boolean dummies to int8.

In [3]:
# Fetch the Adult / Census Income dataset
adult = fetch_openml("adult", version=2, as_frame=True)
X_raw = adult.data
y_raw = adult.target

# Encode label: ">50K" -> 1, "<=50K" -> 0
y = (y_raw == ">50K").astype(int).values

categorical_cols = X_raw.select_dtypes(include=["category", "object"]).columns.tolist()

# Replace "?" markers with NaN, then drop those rows
X_clean = X_raw.copy()
for col in categorical_cols:
    X_clean[col] = X_clean[col].astype(str).replace("?", np.nan)
mask = X_clean.notna().all(axis=1)
X_clean = X_clean[mask].reset_index(drop=True)
y = y[mask.values]

# One-hot encode categoricals
X = pd.get_dummies(X_clean, columns=categorical_cols, drop_first=False)
# Cast bool dummies to int so xgboost / lightgbm are happy
X = X.astype({c: np.int8 for c in X.columns if X[c].dtype == bool})

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

print(f"Training data: {X_train.shape[0]} rows \u00d7 {X_train.shape[1]} columns")
print(f"Testing  data: {X_test.shape[0]} rows \u00d7 {X_test.shape[1]} columns")
print(f"Positive class share (train): {y_train.mean():.3f}")

Training data: 33916 rows × 104 columns
Testing  data: 11306 rows × 104 columns
Positive class share (train): 0.248


## 3. Helper functions

* `run_shap` — creates a `shap.TreeExplainer` (path-dependent, no background),
  runs `num_round` times and reports mean ± std wall-clock time.
  Set `interactions=True` to call `.shap_interaction_values()` instead.
* `run_shapiq` — same pattern for shapiq's `TreeExplainer`.
* `shap_sv_to_array` — normalises shap output to `(n_samples, n_features)`.
* `shapiq_sv_to_array` — normalises shapiq output to `(n_samples, n_features)`.

In [4]:
def run_shap(model, sample, *, interactions, num_round, num_sample,
             background=None):
    """
    Time shap.TreeExplainer `num_round` times. Returns the last result.

    Parameters
    ----------
    model       : trained tree model (RF / XGBoost / LightGBM)
    sample      : DataFrame to explain (first `num_sample` rows are used)
    interactions: if True, call shap_interaction_values() instead of shap_values()
    num_round   : number of timing repetitions
    num_sample  : number of samples to explain per round
    background  : optional background dataset (enables interventional SHAP);
                  pass None (default) for path-dependent mode
    """
    explainer = shap.TreeExplainer(model, background)
    # shap.TreeExplainer works with numpy arrays
    X_arr = sample.iloc[:num_sample].values
    run_time = np.zeros(num_round)
    out = None
    for i in range(num_round):
        start = time.time()
        if interactions:
            out = explainer.shap_interaction_values(X_arr)
        else:
            out = explainer.shap_values(X_arr)
        run_time[i] = time.time() - start
        print(f"  Round {i + 1}: {run_time[i]:.3f} s")
    mode = "background" if background is not None else "path-dependent"
    label = f"shap ({mode})" + (" interactions" if interactions else "")
    print(f"=> {label}: {run_time.mean():.3f} \u00b1 {run_time.std():.3f} s")
    return out, run_time

In [5]:
def run_shapiq(model, sample, *, max_order, n_jobs, num_round, num_sample,
               index=None, class_index=None):
    """Time shapiq's TreeExplainer `num_round` times. Returns the last result."""
    if index is None:
        index = "SV" if max_order == 1 else "k-SII"
    explainer = shapiq.TreeExplainer(
        model=model,
        max_order=max_order,
        min_order=1,
        index=index,
        class_index=class_index,
    )
    X_arr = sample.iloc[:num_sample].values
    run_time = np.zeros(num_round)
    out = None
    for i in range(num_round):
        start = time.time()
        out = explainer.explain_X(X_arr, n_jobs=n_jobs)
        run_time[i] = time.time() - start
        print(f"  Round {i + 1}: {run_time[i]:.3f} s")
    label = f"shapiq (max_order={max_order}, index={index})"
    print(f"=> {label}: {run_time.mean():.3f} \u00b1 {run_time.std():.3f} s")
    return out, run_time

In [6]:
def shap_sv_to_array(shap_out, class_index=None):
    """
    Normalise shap.TreeExplainer SV output to (n_samples, n_features).

    shap.TreeExplainer output:
    - Binary RF / LightGBM (multi-output): list of 2 arrays, each
      (n_samples, n_features). We take element `class_index` (default 1).
    - XGBoost binary (single log-odds output): 2-D array (n_samples, n_features).
    """
    if isinstance(shap_out, list):
        idx = class_index if class_index is not None else 1
        return np.asarray(shap_out[idx])
    return np.asarray(shap_out)


def shapiq_sv_to_array(iv_result, n_features):
    """Convert shapiq per-sample InteractionValues (order 1) to (n_samples, n_features)."""
    if isinstance(iv_result, list):
        rows = []
        for iv in iv_result:
            row = iv.get_n_order_values(1)
            rows.append(np.asarray(row).ravel())
        return np.vstack(rows)
    return np.asarray(iv_result.values)

In [7]:
# Global benchmark settings
NUM_SAMPLE_SV  = 1000   # samples to explain for SV benchmark
NUM_SAMPLE_INT = 100    # samples to explain for interaction benchmark
NUM_ROUND      = 3      # repetitions for mean/std
N_JOBS         = -1     # shapiq: all available cores

## 4. Random Forest (scikit-learn)

`RandomForestClassifier(n_estimators=200, max_depth=8)`.

In [8]:
n_estimators = 200
max_depth    = 8

rf_model = RandomForestClassifier(
    n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=0
)
rf_model.fit(X_train, y_train)

print(f"AUC:      {roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1]):.3f}")
print(f"Accuracy: {accuracy_score(y_test, rf_model.predict(X_test)):.3f}")

AUC:      0.904
Accuracy: 0.848


### 4a. Shapley values — correctness check

Both shap and shapiq (index=`"SV"`) implement the standard path-dependent
TreeSHAP and should agree numerically up to floating-point noise.

In [9]:
# shap (path-dependent, no background data)
sv_shap, _ = run_shap(
    rf_model, X_test, interactions=False,
    num_round=1, num_sample=NUM_SAMPLE_SV,
)

# shapiq (max_order=1, index="SV", class_index=1 for positive class)
sv_shapiq, _ = run_shapiq(
    rf_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

  Round 1: 4.351 s
=> shap (path-dependent): 4.351 ± 0.000 s
  Round 1: 8753.928 s
=> shapiq (max_order=1, index=SV): 8753.928 ± 0.000 s


In [ ]:
# shap (path-dependent, no background data)
sv_shap, _ = run_shap(
    rf_model, X_test, interactions=False,
    num_round=1, num_sample=NUM_SAMPLE_SV,
)

# shapiq (max_order=1, index="SV", class_index=1 for positive class)
sv_shapiq, _ = run_shapiq(
    rf_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

  Round 1: 5.405 s
=> shap (path-dependent): 5.405 ± 0.000 s


In [11]:
# Compare SV values
# shap for RF binary returns a list of 2 arrays; take index 1 (positive class)
sv_shap_arr   = shap_sv_to_array(sv_shap, class_index=1)
sv_shapiq_arr = shapiq_sv_to_array(sv_shapiq, X_test.shape[1])

print(f"shap vs shapiq: max |diff| = {np.max(np.abs(sv_shap_arr - sv_shapiq_arr)):.2e}")
print(f"                mean |diff|= {np.mean(np.abs(sv_shap_arr - sv_shapiq_arr)):.2e}")

ValueError: operands could not be broadcast together with shapes (1000,104,2) (1000,104) 

### 4b. Timing benchmark — Shapley values

In [ ]:
print("--- shap (path-dependent SV) ---")
_, t_rf_shap_sv = run_shap(
    rf_model, X_test, interactions=False,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- shapiq (SV) ---")
_, t_rf_sq_sv = run_shapiq(
    rf_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

### 4c. Pairwise interactions — shap vs shapiq

**Important**: the two libraries use different interaction indices.
shap returns the **TreeSHAP interaction values** (Lundberg et al. 2020),
an $n \times n$ matrix with main effects on the diagonal. shapiq returns
**k-SII** values, which aggregate higher-order terms differently.
The values are not numerically equal, but the timing comparison is valid.

Sample count is reduced from `NUM_SAMPLE_SV` to `NUM_SAMPLE_INT` because
interaction values cost $O(n_{\text{features}})$ more per sample.

In [ ]:
print("--- shap (TreeSHAP interactions) ---")
_, t_rf_shap_int = run_shap(
    rf_model, X_test, interactions=True,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)
print()
print("--- shapiq (k-SII, max_order=2) ---")
_, t_rf_sq_int = run_shapiq(
    rf_model, X_test, max_order=2,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
    class_index=1,
)

## 5. XGBoost

`XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1)`.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1, n_jobs=-1,
    eval_metric="logloss", random_state=0,
)
xgb_model.fit(X_train, y_train)

print(f"AUC:      {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]):.3f}")
print(f"Accuracy: {accuracy_score(y_test, xgb_model.predict(X_test)):.3f}")

### 5a. Shapley values — correctness check

In [ ]:
# XGBoost binary: shap returns a single 2-D array (log-odds space)
sv_shap, _ = run_shap(
    xgb_model, X_test, interactions=False,
    num_round=1, num_sample=NUM_SAMPLE_SV,
)
sv_shapiq, _ = run_shapiq(
    xgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
)

sv_shap_arr   = shap_sv_to_array(sv_shap)
sv_shapiq_arr = shapiq_sv_to_array(sv_shapiq, X_test.shape[1])
print(f"shap vs shapiq: max |diff| = {np.max(np.abs(sv_shap_arr - sv_shapiq_arr)):.2e}")
print(f"                mean |diff|= {np.mean(np.abs(sv_shap_arr - sv_shapiq_arr)):.2e}")

### 5b. Timing — Shapley values

In [ ]:
print("--- shap (path-dependent SV) ---")
_, t_xgb_shap_sv = run_shap(
    xgb_model, X_test, interactions=False,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- shapiq (SV) ---")
_, t_xgb_sq_sv = run_shapiq(
    xgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)

### 5c. Timing — pairwise interactions

In [ ]:
print("--- shap (TreeSHAP interactions) ---")
_, t_xgb_shap_int = run_shap(
    xgb_model, X_test, interactions=True,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)
print()
print("--- shapiq (k-SII, max_order=2) ---")
_, t_xgb_sq_int = run_shapiq(
    xgb_model, X_test, max_order=2,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)

## 6. LightGBM

`LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.1)`.

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1, n_jobs=-1,
    random_state=0, verbosity=-1,
)
lgb_model.fit(X_train, y_train)

print(f"AUC:      {roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1]):.3f}")
print(f"Accuracy: {accuracy_score(y_test, lgb_model.predict(X_test)):.3f}")

### 6a. Shapley values — correctness check

In [ ]:
# LightGBM binary: shap returns a list of 2 arrays; take index 1
sv_shap, _ = run_shap(
    lgb_model, X_test, interactions=False,
    num_round=1, num_sample=NUM_SAMPLE_SV,
)
sv_shapiq, _ = run_shapiq(
    lgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

sv_shap_arr   = shap_sv_to_array(sv_shap, class_index=1)
sv_shapiq_arr = shapiq_sv_to_array(sv_shapiq, X_test.shape[1])
print(f"shap vs shapiq: max |diff| = {np.max(np.abs(sv_shap_arr - sv_shapiq_arr)):.2e}")
print(f"                mean |diff|= {np.mean(np.abs(sv_shap_arr - sv_shapiq_arr)):.2e}")

### 6b. Timing — Shapley values

In [ ]:
print("--- shap (path-dependent SV) ---")
_, t_lgb_shap_sv = run_shap(
    lgb_model, X_test, interactions=False,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- shapiq (SV) ---")
_, t_lgb_sq_sv = run_shapiq(
    lgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

### 6c. Timing — pairwise interactions

In [ ]:
print("--- shap (TreeSHAP interactions) ---")
_, t_lgb_shap_int = run_shap(
    lgb_model, X_test, interactions=True,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)
print()
print("--- shapiq (k-SII, max_order=2) ---")
_, t_lgb_sq_int = run_shapiq(
    lgb_model, X_test, max_order=2,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)

## 7. Summary table

`SV` rows are over `NUM_SAMPLE_SV` samples; `Interaction` rows over `NUM_SAMPLE_INT`.

In [ ]:
def fmt(t):
    return f"{t.mean():.3f} \u00b1 {t.std():.3f}"

summary = pd.DataFrame(
    {
        "RF": [
            fmt(t_rf_shap_sv),  fmt(t_rf_sq_sv),
            fmt(t_rf_shap_int), fmt(t_rf_sq_int),
        ],
        "XGBoost": [
            fmt(t_xgb_shap_sv),  fmt(t_xgb_sq_sv),
            fmt(t_xgb_shap_int), fmt(t_xgb_sq_int),
        ],
        "LightGBM": [
            fmt(t_lgb_shap_sv),  fmt(t_lgb_sq_sv),
            fmt(t_lgb_shap_int), fmt(t_lgb_sq_int),
        ],
    },
    index=pd.MultiIndex.from_tuples(
        [
            ("SV",          f"shap path-dep    ({NUM_SAMPLE_SV} samples)"),
            ("SV",          f"shapiq SV        ({NUM_SAMPLE_SV} samples)"),
            ("Interaction", f"shap TreeSHAP    ({NUM_SAMPLE_INT} samples)"),
            ("Interaction", f"shapiq k-SII     ({NUM_SAMPLE_INT} samples)"),
        ],
        names=["Quantity", "Method"],
    ),
)
summary

## 8. Caveats and things to try

* **Perturbation mode**: the timings above use path-dependent perturbation.
  To benchmark interventional SHAP with shap, pass
  `background=shap.sample(X_train, 100)` to `run_shap`.

* **Interaction index choice for shapiq**: we used `k-SII`. To benchmark
  `SII` (mathematically closest to shap's interaction matrix) or `STII`,
  pass `index="SII"` or `index="STII"` to `run_shapiq`.

* **Higher-order interactions**: shapiq supports order-3+ interactions
  (TreeSHAP-IQ's main advantage over shap). shap is limited to pairwise
  (order-2) interactions. Bump `max_order=3` in `run_shapiq` to explore.

* **Sample sizes**: adjust `NUM_SAMPLE_SV` and `NUM_SAMPLE_INT` as needed.
  Interaction values are significantly more expensive for both libraries.